# 09. 생성 모델

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/2038prob/defense-ai-textbook-code/blob/main/notebooks/%EC%8B%A4%EC%8A%B5_09_%EC%83%9D%EC%84%B1_%EB%AA%A8%EB%8D%B8.ipynb)

「국방 인공지능」 실습 코드 — 이 장의 파이썬 코드를 순서대로 모았습니다. **맨 위 setup 셀부터 위에서 아래로** 실행하세요(앞 셀에서 정의한 변수를 뒤 셀이 이어 씁니다). ⚠️ 표시가 붙은 셀은 개념 설명용 조각으로, 그대로는 실행되지 않을 수 있습니다.

전체 저장소: https://github.com/2038prob/defense-ai-textbook-code


In [ ]:
# === 실습 환경 준비 (Colab/로컬 공통) — 가장 먼저 실행하세요 ===
# 필요한 패키지를 설치하고(이미 있으면 건너뜀) 공통 유틸을 정의합니다.
import importlib.util, subprocess, sys
for pkg in ["torch", "torchvision", "numpy", "matplotlib", "scipy"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

import random, numpy as np, torch

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", device)

## 실습: VAE와 조건부 GAN

### 🔬 실습 — VAE 구현과 학습

① **VAE 모델.** 인코더가 \(\mu,\log\sigma^2\)을 출력하고, 재파라미터화로 \(z\)를 샘플링한다.


In [ ]:
import torch, torch.nn as nn

class VAE(nn.Module):
    def __init__(self, in_dim=784, h=400, z_dim=20):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, h)
        self.fc_mu     = nn.Linear(h, z_dim)
        self.fc_logvar = nn.Linear(h, z_dim)
        self.dec = nn.Sequential(nn.Linear(z_dim, h), nn.ReLU(),
                                 nn.Linear(h, in_dim), nn.Sigmoid())
    def encode(self, x):
        h = torch.relu(self.fc1(x)); return self.fc_mu(h), self.fc_logvar(h)
    def reparam(self, mu, logvar):
        std = torch.exp(0.5 * logvar); eps = torch.randn_like(std)
        return mu + std * eps                       # 재파라미터화 트릭
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparam(mu, logvar)
        return self.dec(z), mu, logvar

② **ELBO 손실함수.** 재구성(BCE) 항에 KL 닫힌 형태를 더한다.


In [ ]:
import torch.nn.functional as F
def vae_loss(recon, x, mu, logvar):
    recon_loss = F.binary_cross_entropy(recon, x, reduction="sum")
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + kl

③ **학습 루프.** 표준 학습 루프로 ELBO 손실을 최소화한다.


In [ ]:
from torchvision import datasets, transforms
train_loader = torch.utils.data.DataLoader(
    datasets.MNIST("data", train=True, download=True,
                   transform=transforms.ToTensor()),   # [0,1] 정규화
    batch_size=128, shuffle=True)

torch.manual_seed(0)                       # 재현성(2장)
model = VAE(); optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
for epoch in range(10):
    total = 0.0
    for x, _ in train_loader:
        x = x.view(x.size(0), -1)        # (B,1,28,28) -> (B,784)
        recon, mu, logvar = model(x)
        loss = vae_loss(recon, x, mu, logvar)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total += loss.item()
    print(f"ep{epoch+1:02d} -ELBO(샘플당) "
          f"{total/len(train_loader.dataset):.1f}")

**실행 결과 예시**

```
ep01 -ELBO(샘플당) 163.3
ep05 -ELBO(샘플당) 109.6
ep10 -ELBO(샘플당) 106.1
생성 샘플: torch.Size([16, 784])
```

(매 에포크 한 줄씩 출력되며 위는 그중 세 줄이다. 본문 코드를 시드 0으로 그대로 실행한 결과이다.)


④ **새 샘플 생성.** 사전 분포에서 \(z\)를 뽑아 디코더로 복원한다.


In [ ]:
model.eval()
with torch.no_grad():
    z = torch.randn(16, 20)              # N(0,I)에서 샘플링
    samples = model.dec(z)               # (16, 784) 새 이미지
print("생성 샘플:", samples.shape)

### 🔬 실습 — 조건부 GAN: 원하는 클래스를 지정해 생성하기

① **모델.** 레이블을 임베딩해 입력에 이어 붙인다(concatenate).


In [ ]:
import torch, torch.nn as nn

class CGenerator(nn.Module):
    def __init__(self, z_dim=64, n_cls=10, emb=16):
        super().__init__()
        self.label_emb = nn.Embedding(n_cls, emb)     # 레이블 -> 벡터
        self.net = nn.Sequential(
            nn.Linear(z_dim + emb, 256), nn.ReLU(),
            nn.Linear(256, 512), nn.ReLU(),
            nn.Linear(512, 784), nn.Tanh())           # 출력 [-1,1]
    def forward(self, z, y):
        zy = torch.cat([z, self.label_emb(y)], dim=1) # 노이즈+조건 결합
        return self.net(zy)

class CDiscriminator(nn.Module):
    def __init__(self, n_cls=10, emb=16):
        super().__init__()
        self.label_emb = nn.Embedding(n_cls, emb)
        self.net = nn.Sequential(
            nn.Linear(784 + emb, 512), nn.LeakyReLU(0.2),
            nn.Linear(512, 256), nn.LeakyReLU(0.2),
            nn.Linear(256, 1))                        # 진짜/가짜 로짓
    def forward(self, x, y):
        xy = torch.cat([x, self.label_emb(y)], dim=1) # 이미지+조건 결합
        return self.net(xy)

② **번갈아 학습.** 9.3절의 minimax를 그대로 쓰고 조건만 더한다.


In [ ]:
from torchvision import datasets, transforms
tfm = transforms.Compose([transforms.ToTensor(),
                          transforms.Normalize(0.5, 0.5)])  # [-1,1] 정규화
train_loader = torch.utils.data.DataLoader(
    datasets.MNIST("data", train=True, download=True, transform=tfm),
    batch_size=128, shuffle=True)

torch.manual_seed(0)                       # 재현성(2장)
G, D = CGenerator(), CDiscriminator()
opt_g = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_d = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
bce = nn.BCEWithLogitsLoss()

for epoch in range(10):
    for x, y in train_loader:
        x = x.view(x.size(0), -1)           # (B,1,28,28) -> (B,784)
        B = x.size(0)
        real, fake = torch.ones(B, 1), torch.zeros(B, 1)
        # --- 판별기: 진짜는 진짜로, 가짜는 가짜로 ---
        z = torch.randn(B, 64)
        x_fake = G(z, y).detach()           # G 고정(detach)하고 D만 학습
        d_loss = bce(D(x, y), real) + bce(D(x_fake, y), fake)
        opt_d.zero_grad(); d_loss.backward(); opt_d.step()
        # --- 생성기: 가짜를 진짜로 속이도록 ---
        z = torch.randn(B, 64)
        g_loss = bce(D(G(z, y), y), real)   # D를 속이는 방향
        opt_g.zero_grad(); g_loss.backward(); opt_g.step()
    print(f"ep{epoch+1:02d} d_loss {d_loss.item():.2f} "
          f"| g_loss {g_loss.item():.2f}")

③ **원하는 숫자만 생성.** 조건이 실제로 작동하는지 확인한다.


In [ ]:
G.eval()
with torch.no_grad():
    z = torch.randn(8, 64)
    y = torch.full((8,), 7)                 # "7"만 8장 생성
    samples = G(z, y)

**실행 결과 예시**

```
ep10 d_loss 0.31 | g_loss 2.68 — 지정한 7 형태의 샘플이 생성된다
```

(본문 코드를 시드 0으로 그대로 실행한 결과이다. 에포크를 10으로 둔 데에는 이유가 있다. 이 단순한 MLP 구성으로 30 에포크까지 돌리면 판별기가 압승해 d_loss가 0으로 떨어지고 생성기가 더 배우지 못하는 균형 실패가 흔히 나타난다. 9.3절에서 경고한 바로 그 위험이니 직접 늘려서 확인해 보라.)


### 🔬 실습(노트북 보강) — 소형 확산 모델(DDPM): 노이즈에서 숫자를 만든다

9.4절의 DDPM을 MNIST에서 직접 학습해 본다. 본문에는 없는 노트북 보강 실습이다.

① **노이즈 스케줄과 노이즈 예측기.** 단계 수 \(T=200\)에서 \(\beta_t\)를 선형으로 늘리고 누적곱 \(\bar\alpha_t\)를 미리 구한다. 노이즈 예측기는 해상도를 28→14→7로 줄였다가 되돌리는 작은 U-Net이며, 시점 \(t\)는 사인파 임베딩으로 각 블록에 더한다.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, math
from torchvision import datasets, transforms
device = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(0)                                   # 재현성(2장)
tfm = transforms.Compose([transforms.ToTensor(),
                          transforms.Normalize(0.5, 0.5)])   # [-1,1] 정규화
loader = torch.utils.data.DataLoader(
    datasets.MNIST("data", train=True, download=True, transform=tfm),
    batch_size=128, shuffle=True)

T = 200                                                # 확산 단계 수
betas = torch.linspace(1e-4, 0.02, T)
alphas = 1 - betas
abar = torch.cumprod(alphas, 0)                        # 누적곱 \bar{alpha}_t

def t_embed(t, dim=32):                                # 사인파 시간 임베딩(7장 위치 인코딩과 같은 발상)
    f = torch.exp(-math.log(10000) * torch.arange(dim // 2, device=t.device) / (dim // 2))
    a = t[:, None].float() * f[None]
    return torch.cat([a.sin(), a.cos()], 1)

class Block(nn.Module):                                # 합성곱 2층 + 시간 임베딩 주입
    def __init__(self, cin, cout):
        super().__init__()
        self.c1 = nn.Conv2d(cin, cout, 3, padding=1)
        self.c2 = nn.Conv2d(cout, cout, 3, padding=1)
        self.t = nn.Linear(32, cout)
    def forward(self, x, te):
        h = F.silu(self.c1(x)) + self.t(te)[:, :, None, None]
        return F.silu(self.c2(h))

class TinyUNet(nn.Module):                             # 28 -> 14 -> 7 -> 14 -> 28 (4장 수용영역 확대)
    def __init__(self):
        super().__init__()
        self.d1, self.d2, self.mid = Block(1, 32), Block(32, 64), Block(64, 64)
        self.u2, self.u1 = Block(128, 64), Block(96, 32)
        self.out = nn.Conv2d(32, 1, 1)
    def forward(self, x, t):
        te = t_embed(t)
        h1 = self.d1(x, te)                            # 28x28
        h2 = self.d2(F.avg_pool2d(h1, 2), te)          # 14x14
        m = self.mid(F.avg_pool2d(h2, 2), te)          # 7x7
        u = self.u2(torch.cat([F.interpolate(m, scale_factor=2), h2], 1), te)
        u = self.u1(torch.cat([F.interpolate(u, scale_factor=2), h1], 1), te)
        return self.out(u)                             # 예측한 노이즈 eps

model = TinyUNet().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

② **학습.** 깨끗한 영상 \(x_0\)에 시점 \(t\)의 노이즈를 한 번에 섞어 \(x_t=\sqrt{\bar\alpha_t}\,x_0+\sqrt{1-\bar\alpha_t}\,\epsilon\)을 만들고, 섞은 노이즈 \(\epsilon\)을 맞히도록 평균제곱오차로 학습한다.

In [ ]:
for epoch in range(10):
    total, n = 0.0, 0
    for x, _ in loader:
        t = torch.randint(0, T, (x.size(0),))
        eps = torch.randn_like(x)
        ab = abar[t][:, None, None, None]
        x_t = ab.sqrt() * x + (1 - ab).sqrt() * eps    # 정방향: q(x_t | x_0)
        loss = F.mse_loss(model(x_t.to(device), t.to(device)), eps.to(device))  # 노이즈 예측 손실
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item() * x.size(0); n += x.size(0)
    print(f"ep{epoch+1} loss {total/n:.4f}")

③ **샘플링.** 순수한 노이즈에서 출발해 \(t=T-1\)부터 0까지, 예측한 노이즈를 한 단계씩 걷어낸다.

In [ ]:
@torch.no_grad()
def sample(k=16):                                      # 역방향: 노이즈에서 한 단계씩 걷어낸다
    x = torch.randn(k, 1, 28, 28)
    for t in reversed(range(T)):
        tt = torch.full((k,), t)
        eps = model(x.to(device), tt.to(device)).cpu()
        x = (x - betas[t] / (1 - abar[t]).sqrt() * eps) / alphas[t].sqrt()
        if t > 0:
            x = x + betas[t].sqrt() * torch.randn_like(x)
    return x.clamp(-1, 1)

imgs = sample()
print("생성 샘플:", imgs.shape)
import matplotlib.pyplot as plt, torchvision
grid = torchvision.utils.make_grid((imgs + 1) / 2, nrow=8)
plt.imshow(grid.permute(1, 2, 0)); plt.axis("off"); plt.show()

**실행 결과 예시**

```
ep1 loss 0.1934
ep5 loss 0.0631
ep10 loss 0.0536
생성 샘플: torch.Size([16, 1, 28, 28])
```

(매 에포크 한 줄씩 출력되며 위는 그중 세 줄이다. 코드를 CPU에서 시드 0으로 그대로 실행한 결과이며, GPU에서는 소수점 아래 값이 조금 달라질 수 있다. CPU로는 25분 가까이 걸리므로 Colab의 GPU 런타임을 권한다. 10에포크로는 16개 샘플 가운데 일부만 숫자 모양이 뚜렷하고, 에포크나 채널 수를 늘리면 나아진다.)

---

실습 코드 저장소: https://github.com/2038prob/defense-ai-textbook-code  ·  교재 「국방 인공지능」
